In [1]:
import librosa
import librosa.display
import soundfile as sf
import pretty_midi
import fast_bss_eval
from scipy.io import wavfile

from musicProcessing import MusicProcessingProvider as mp

from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import numpy as np

from pandas import DataFrame
import pandas as pd

In [2]:
def get_source_activity(midi_path, sr, hop_length, max_frames):
    # loads individual source midi and returns a np.array with all the active time frame indexes
    pm = pretty_midi.PrettyMIDI(midi_path)
    active_frames = np.asarray([], dtype=int)
    for instrument in pm.instruments:
        for note in instrument.notes:
            start_frame = np.floor(note.start * sr / hop_length).astype(int)
            end_frame = np.floor(note.end * sr / hop_length).astype(int)
            active_frames = np.union1d(active_frames, np.arange(start_frame, end_frame))
    # clip to max_frames
    active_frames = active_frames[active_frames < max_frames]
    return active_frames

In [ ]:
def eval(estimated_path, reference_path):
    _, ref = wavfile.read(reference_path)
    _, est = wavfile.read(estimated_path)
    
    sdr, sir, sar, perm = fast_bss_eval.bss_eval_sources(ref.T, est.T)
    
    return sdr, sir, sar

In [4]:
baby_slakh_path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/babyslakh_16k/"
results_path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/experiments/results/exp3/"
repetiveness_scores_path = "/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/repetiveness.csv"

repetiveness_df = pd.read_csv(repetiveness_scores_path)
#drop all cols but track_id, stem_id and score
repetiveness_df = repetiveness_df[["track_id", "stem_id", "score"]]
# sort by track name then by source stem
repetiveness_df = repetiveness_df.sort_values(by=["track_id", "stem_id"])

nfft = 2048
win_length = 2048
hop_length = 512
window_type = "hann"

repetiveness_df.head()

,track_id,stem_id,score
118,Track00001,S00,0.782380
86,Track00001,S01,0.815883
159,Track00001,S02,0.712629
69,Track00001,S03,0.843499
129,Track00001,S04,0.761250


In [5]:
tracks = os.listdir(baby_slakh_path)
tracks.sort()

results = DataFrame(columns=["track_id", "stem_id", "sdr", "sir", "sar", "repetiveness_score"])

for track in tqdm(tracks):
    track_path = os.path.join(baby_slakh_path, track)
    
    # create folder for results if it doesn't exist
    track_results_path = os.path.join(results_path, track)
    os.makedirs(track_results_path, exist_ok=True)

    mixture_path = os.path.join(track_path, "mix.wav")
    stems_path = os.path.join(track_path, "stems")
    midi_path = os.path.join(track_path, "MIDI")
    stem_wave_paths = [os.path.join(stems_path, f) for f in os.listdir(stems_path) if f.endswith(".wav")]
    stem_midi_paths = [os.path.join(midi_path, f) for f in os.listdir(midi_path) if f.endswith(".mid")]
    
    music_processor = mp(path=mixture_path, duration=None, nfft=nfft, hop_length=hop_length, win_length=win_length, window_type=window_type)
    
    for stem_wave_path, stem_midi_path in zip(stem_wave_paths, stem_midi_paths):
        active_frames = get_source_activity(stem_midi_path, music_processor.sr, hop_length, music_processor.magnitude.shape[1])
        
        # perform median filtering on the active frames
        W = music_processor.median_filter(frames=active_frames)
        mask = music_processor.calc_mask(W)

        source_data = music_processor.apply_mask(music_processor.stft, mask)
        source_data = music_processor.calc_iSTFT(source_data, hop_length=hop_length, win_length=win_length, window_type=window_type)
        
        # save the estimated source
        stem_name = os.path.basename(stem_wave_path).replace(".wav", "")
        estimated_path = os.path.join(track_results_path, f"{stem_name}_estimated.wav")
        sf.write(estimated_path, source_data, music_processor.sr)
        
        # evaluate the estimated source against the original stem
        sdr, sir, sar = eval(estimated_path, stem_wave_path)
        
        # add repetiveness score
        track_id = track
        stem_id = stem_name
        repetiveness_score = repetiveness_df[(repetiveness_df["track_id"] == track_id) & (repetiveness_df["stem_id"] == stem_id)]["score"].values
        repetiveness_score = repetiveness_score[0] if len(repetiveness_score) > 0 else None
        
        # save results to dataframe
        results = results.append({
            "track_id": track_id,
            "stem_id": stem_id,
            "sdr": sdr,
            "sir": sir,
            "sar": sar,
            "repetiveness_score": repetiveness_score
        }, ignore_index=True)
results.to_csv(os.path.join(results_path, "evaluation_results.csv"), index=False)

  0%|          | 0/20 [00:00<?, ?it/s]/home/bordduk69/Nextcloud/NTNU/10. Vår2026/Master/Code/REBEAT/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/20 [00:04<?, ?it/s]


ValueError: einstein sum subscripts string contains too many subscripts for operand 0